In [33]:
import pandas as pd
from ingest import build_index, load_faq_data
from tqdm.auto import tqdm

In [2]:
df_ground_truth = pd.read_csv("data/ground-truth-data.csv")

In [3]:
ground_truth = df_ground_truth.to_dict("records")

In [4]:
documents_llm = load_faq_data()

In [5]:
index = build_index(documents_llm)

In [18]:
gt_ids = {q["document"] for q in ground_truth}
index_ids = {doc["id"] for doc in documents_llm}
print(f"missing: {len(gt_ids - index_ids)} / {len(gt_ids)}")

missing: 6 / 1244


In [6]:
index.search("What is the course about?")

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [7]:
def text_search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [8]:
q = ground_truth[1]

In [9]:
q

{'question': 'Where can I find the homework assignments and the submission form for this cohort?',
 'document': '0e38656cfb'}

In [10]:
results = text_search(q["question"])
results

[{'id': '53f15299b6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'Where can I find the homework questions?',
  'answer': 'Homework links are available in the course GitHub repo and in the course management platform.\n\nFor the 2026 Module 1 homework, use:\n\n- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)\n- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md)\n\nThe course platform is useful for submission and deadlines, but the GitHub homework instructions often contain important extra context.'},
 {'id': '0190b37350',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where do I find the updated homework playlist and cohort-specific info for the MLOps course?',
  'answer': "All cohort-specific information (homework links, video playlist, deadlines, Slack channel) liv

In [11]:
doc_id = "53f15299b6"

In [12]:
for d in results:
    print(f"{d["id"]} == {doc_id}: {d["id"] == doc_id}")

53f15299b6 == 53f15299b6: True
0190b37350 == 53f15299b6: False
a9353fadfe == 53f15299b6: False
90b9e103c7 == 53f15299b6: False
50ebc48eea == 53f15299b6: False


In [19]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query = q["question"])
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [20]:
compute_relevance_text(q)

[0, 0, 0, 0, 0]

In [30]:
q = ground_truth[15]
compute_relevance_text(q)

[0, 1, 0, 0, 0]

In [31]:
q = ground_truth[70]
compute_relevance_text(q)

[1, 0, 0, 0, 0]

In [32]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance_text(q))
        
    return relevance_total

In [34]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/6220 [00:00<?, ?it/s]

In [38]:
sample = relevance[:15]

In [36]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query = q["question"])
    relevance = []
    
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [37]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
        
    return relevance_total

In [39]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
            
    return cnt / len(relevance)

In [40]:
hit_rate(sample)

0.5333333333333333

In [ ]:
def mean_reciprocal_rank(relevance: list[list[int]]) -> float:
    total_rr = 0.0
    for line in relevance:
        if 1 in line:
            total_rr += 1 / (line.index(1) + 1)
        # else: contributes 0, but still counts toward the denominator
    return total_rr / len(relevance)

In [44]:
mean_reciprocal_rank(sample)

0.38333333333333336